# Local Data Agent - temporary free hosting on a Colab GPU

Runs the whole stack on this notebook's VM - MySQL with the `employees` sample database, Ollama on the T4 GPU
(`qwen2.5-coder:7b` for SQL, `qwen3:8b` thinking model for answers and the Expert AI), the trained ML models and the
Streamlit app - and opens a public link you can share.

**Before you run:** *Runtime -> Change runtime type -> T4 GPU*. Then *Runtime -> Run all*. First run takes ~10 minutes
(downloading ~10 GB of models). The link stays up while this notebook is running (Colab stops it after ~90 min idle
or ~12 h); each run gets a new random URL.


In [ ]:
#@title 1. GPU check + get the code
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "No GPU: choose Runtime -> Change runtime type -> T4 GPU"
import os
REPO = "https://github.com/k642512255032-max/database-agent.git"
if not os.path.isdir("/content/database-agent"):
    !git clone -q {REPO} /content/database-agent
%cd /content/database-agent
!git pull -q


In [ ]:
#@title 2. Install everything (MySQL + employees DB, Ollama + models, app deps, trained models) - ~10 min
!bash deploy/colab/bootstrap.sh


In [ ]:
#@title 3. Start the app and print the public link
!bash deploy/colab/serve.sh


In [ ]:
#@title 4. Keep the session alive (leave this running while people use the link; stop it to end hosting)
import re, subprocess, time
url = ""
try:
    url = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/tmp/agent-logs/cloudflared.log").read()).group(0)
except Exception:
    pass
print("Sharing:", url or "(see cell 3)")
while True:
    alive = subprocess.run(["pgrep", "-f", "streamlit run app.py"], capture_output=True).returncode == 0
    print(time.strftime("%H:%M:%S"), "app running" if alive else "APP STOPPED - re-run cell 3", "|", url, flush=True)
    time.sleep(300)


### Notes
* **Netlify publishing (Dashboards page)** is off here - no `NETLIFY_AUTH_TOKEN` is set on the VM. Add one to `.env` only
  if you want people to publish dashboards from this link.
* Everyone with the link can query the sample database (read-only user) and use the models. Nothing else on the VM
  is exposed.
* Logs: `/tmp/agent-logs/` (`streamlit.log`, `ollama.log`, `cloudflared.log`, `train.log`).
* Slow first answer: the models load into the GPU on first use (~30 s), then a chat turn with the Expert AI takes
  roughly 20-60 s on a T4.
